# Pipeline1

Proceso para usar el pipeline:

- Mandar el correo, descargarlo y guardarlo en la carpeta Finanzas Power BI. Este proceso se hace el domíngo
- Correr el pipeline. 
    - En caso de que haya incosistencias en el excel de la app, se va a detener y no guardará las nuevas transacciones en la base de datos y habrá que abrir el excel de la app y corregirlo manualmente
    - En caso de que todo esté bien con las categorías, se agregarán a la base de datos
- En caso de que se corra el pipeline y haya algun tipo de error será necesario borrar lo agregado en la base de datos, guardar el excel y volver a correr para que no se sobreescriba
    

In [1]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import unicodedata
from datetime import datetime, timedelta
import os
from openpyxl import load_workbook


# =========================
# 2. CONFIGURACIÓN Y CARGA
# =========================
ruta = "Base_de_datos_finanzas_personales.xlsx"

df_base = pd.read_excel(ruta)

# Lectura manual
df_app = pd.read_excel("Registro Contable_31-5-26.xlsx")


# =========================
# 3. VALIDACIÓN INICIAL
# =========================
print("Tamaño del DataFrame base:", df_base.shape)
print("Tamaño del DataFrame de la app:", df_app.shape)


# =========================
# 4. SELECCIÓN DE COLUMNAS
# =========================
columns = ["Fecha", "Importe", "Categoría", "Ingreso/Gasto", "Cuenta", "Nota", "Descripción"]
existing_columns = [col for col in columns if col in df_app.columns]
filtered_df = df_app[existing_columns].copy()


# =========================
# 5. LIMPIEZA Y NORMALIZACIÓN
# =========================

# Fecha
if "Fecha" in filtered_df.columns:
    filtered_df["Fecha"] = pd.to_datetime(filtered_df["Fecha"], errors="coerce").dt.normalize()

# Importe
if "Importe" in filtered_df.columns:
    filtered_df["Importe"] = filtered_df["Importe"].fillna(0).astype(int)

# Función para limpiar texto
def remove_accents(text):
    if pd.isna(text):
        return text
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFKD', text)
    return "".join([c for c in text if not unicodedata.combining(c)])

# Aplicar limpieza
categorical_cols = ["Categoría", "Ingreso/Gasto", "Cuenta", "Nota", "Descripción"]
for col in categorical_cols:
    if col in filtered_df.columns:
        filtered_df[col] = filtered_df[col].apply(remove_accents)


# =========================
# 6. RENOMBRE DE COLUMNAS
# =========================
rename_map = {
    "Categoría": "categoria_id_aux",
    "Ingreso/Gasto": "tipo_id_aux",
    "Cuenta": "cuenta_id_aux",
    "Nota": "detalle_id_aux",
    "Descripción": "control_id_aux",
    "Importe": "monto",
    "Fecha": "fecha_id"
}

final_app = filtered_df.rename(columns=rename_map)


# =========================
# 7. MAPEOS
# =========================
categoria_map = {
    "despensa personal": 1,
    "pagos": 2,
    "inversiones": 3,
    "comida": 4,
    "despensa familiar": 5,
    "medico y salud": 6,
    "entretenimiento": 7,
    "ropa y cuidados": 8,
    "educacion": 9,
    "transporte": 10,
    "regalos": 11,
    "salario": 12,
    "personal": 13,
    "otro": 14
}

tipo_map = {
    "gasto": 1,
    "ingreso": 2,
    "desconocido": 0
}

cuenta_map = {
    "bbva": 1,
    "efectivo": 2,
    "tarjeta credito (nu)": 3,
    "hsbc": 4,
    "mercado pago": 5,
    "didi cuenta": 6,
    "tarjeta credito (vexi)": 7,
    "up si vale": 8,
    "desconocido": 0,
    "revolut": 9
}

detalle_map = {
    "celular": 1,
    "saldo": 2,
    "contrato": 3,
    "trabajo": 4,
    "tamales": 5,
    "pelo": 6,
    "ventilador": 7,
    "nachos": 8,
    "pastel": 9,
    "mercado pago": 10,
    "didi": 11,
    "tienda": 12,
    "mama": 13,
    "despensa familiar": 14,
    "pasaje yeendy": 15,
    "agua": 16,
    "cerveza": 17,
    "tianguis": 18,
    "clases": 19,
    "udemy": 20,
    "salario papa": 21,
    "gas": 22,
    "comida mama": 23,
    "desconocido": 24,
    "youtube premium": 25,
    "google drive": 26,
    "dori": 27,
    "despensa personal": 28,
    "didi inversion": 29,
    "one drive": 30,
    "gta6": 31,
    "semana santa": 32,
    "comida rapida": 33,
    "regalo": 34,
    "veterinario": 35,
    "transporte escuela": 36,
    "revolut": 37,
    "yeen": 38,
    "medicamento": 39,
    "monitor": 40,
    "chat gpt": 41,
    "pan": 42,
    "papeleria": 43,
    "coctel": 44,
    "negocio comida local": 45,
    "pizza": 46,
    "utilidades": 47,
    "dama": 48,
    "teclado": 49
}

control_map = {
    "mensuales": 1,
    "variables": 2,
    "inversion": 3,
    "desconocido": 0,
    "ingreso": 4
}


# =========================
# 8. APLICAR MAPEOS
# =========================
if "categoria_id_aux" in final_app.columns:
    final_app["categoria_id"] = final_app["categoria_id_aux"].map(categoria_map)

if "tipo_id_aux" in final_app.columns:
    final_app["tipo_id"] = final_app["tipo_id_aux"].map(tipo_map)

if "cuenta_id_aux" in final_app.columns:
    final_app["cuenta_id"] = final_app["cuenta_id_aux"].map(cuenta_map)

if "detalle_id_aux" in final_app.columns:
    final_app["detalle_id"] = final_app["detalle_id_aux"].map(detalle_map)

if "control_id_aux" in final_app.columns:
    final_app["control_id"] = final_app["control_id_aux"].map(control_map)


# =========================
# 9. VALIDACIÓN DE MAPEOS
# =========================
def revisar_no_mapeados(df, col_aux, col_id):
    if col_aux in df.columns and col_id in df.columns:
        faltantes = df[df[col_aux].notna() & df[col_id].isna()][col_aux].unique()
        if len(faltantes) > 0:
            raise ValueError(f"❌ Valores no mapeados en {col_aux}: {faltantes}")

revisar_no_mapeados(final_app, "categoria_id_aux", "categoria_id")
revisar_no_mapeados(final_app, "tipo_id_aux", "tipo_id")
revisar_no_mapeados(final_app, "cuenta_id_aux", "cuenta_id")
revisar_no_mapeados(final_app, "detalle_id_aux", "detalle_id")
revisar_no_mapeados(final_app, "control_id_aux", "control_id")


# =========================
# 10. ALINEACIÓN CON BASE
# =========================
for col in df_base.columns:
    if col not in final_app.columns:
        final_app[col] = None

final_app = final_app[df_base.columns]


# =========================
# 11. CONCATENACIÓN
# =========================
df_total = pd.concat([df_base, final_app], ignore_index=True)


# =========================
# 12. GENERACIÓN DE IDs
# =========================
max_id = df_base["id_transaccion"].dropna().astype(int).max()

mask_new = df_total["id_transaccion"].isna()

df_total.loc[mask_new, "id_transaccion"] = range(
    max_id + 1,
    max_id + 1 + mask_new.sum()
)


# =========================
# 13. ORDENAMIENTO
# =========================
df_total["fecha_id"] = pd.to_datetime(df_total["fecha_id"], errors="coerce")
df_total = df_total.sort_values(by="fecha_id").reset_index(drop=True)

print("Tamaño del DataFrame combinado:", df_total.shape)


# =========================
# 14. RESUMEN SEMANAL
# =========================
"""fecha_min = df_total.loc[mask_new, "fecha_id"].min()
fecha_max = df_total.loc[mask_new, "fecha_id"].max()

inicio_semana = fecha_min - timedelta(days=fecha_min.weekday())
fin_semana = fecha_max + timedelta(days=(6 - fecha_max.weekday()))

print(
    "Numero de transacciones nuevas:",
    mask_new.sum(),
    "del lunes",
    inicio_semana.strftime("%Y-%m-%d"),
    "al domingo",
    fin_semana.strftime("%Y-%m-%d")
)"""


# =========================
# 15. VALIDACIÓN DE DUPLICADOS
# =========================
duplicados = df_base[df_base["id_transaccion"].duplicated(keep=False)]

if not duplicados.empty:
    print("⚠️ IDs duplicados encontrados:")
    print(duplicados.sort_values("id_transaccion"))
else:
    print("✅ No hay IDs duplicados")


# =========================
# 16. EXPORTACIÓN A EXCEL
# =========================







# =========================
# 16. EXPORTACIÓN A EXCEL
# =========================
with pd.ExcelWriter(
    ruta,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    df_total.to_excel(writer, index=False, sheet_name="Transacciones_fact")
    
    ws = writer.sheets["Transacciones_fact"]
    
    from openpyxl.styles import numbers

    for cell in ws["B"]:
        cell.number_format = "DD/MM/YYYY"
    
    # Ancho de columnas
    ws.column_dimensions['A'].width = 13
    ws.column_dimensions['B'].width = 13
    ws.column_dimensions['C'].width = 11
    ws.column_dimensions['D'].width = 8
    ws.column_dimensions['E'].width = 12
    ws.column_dimensions['F'].width = 14
    ws.column_dimensions['G'].width = 14
    ws.column_dimensions['H'].width = 12
    ws.column_dimensions['I'].width = 3
    ws.column_dimensions['J'].width = 20
    ws.column_dimensions['K'].width = 15
    ws.column_dimensions['L'].width = 15
    ws.column_dimensions['M'].width = 15
    ws.column_dimensions['N'].width = 17

Tamaño del DataFrame base: (381, 14)
Tamaño del DataFrame de la app: (14, 11)
Tamaño del DataFrame combinado: (395, 14)
✅ No hay IDs duplicados
